<a href="https://colab.research.google.com/github/binhvd/AI-Vision/blob/master/05_WildLife_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Wildlife Object Localization with a CNN

This notebook teaches the core ideas behind **object detection** by building a simpler cousin of it: a convolutional neural network (CNN) that looks at a photo containing a single African animal and predicts a **bounding box** around it.

We will:
1. Download a small dataset of animal photos with YOLO-style bounding-box labels.
2. Build a `tf.data` input pipeline that loads images and their boxes.
3. Train a CNN to predict box coordinates directly (**bounding-box regression**).
4. Evaluate the model with **Intersection over Union (IoU)**, the standard metric used in real object detectors.
5. Compare predictions to ground truth, and discuss how production detectors (YOLO, SSD, Faster R-CNN) extend these same ideas to *multiple*, *unknown*, and *classified* objects per image.

> **Note:** Because each image here contains exactly one labeled animal, this notebook only predicts *where* the animal is (its box). Real object detectors additionally predict *what* it is (its class) and can find *many* objects in one image. We revisit this in the wrap-up.

In [ ]:
!pip install opencv-python-headless

## 1. Download the dataset

The dataset contains photos of four African animals — buffalo, elephant, rhino, and zebra — stored as `temp/<class>/<id>.jpg`. Each image has a matching `temp/<class>/<id>.txt` label file in **YOLO format**:

```
<class_id> <x_center> <y_center> <width> <height>
```

All four numbers are normalized to `[0, 1]` relative to the image size, so the labels work for any image resolution.

In [ ]:
!test -f animal.zip || wget -q https://storage.googleapis.com/adsa-data/african-wildlife/animal.zip
!test -d temp || unzip -q animal.zip -d temp

## 2. List and shuffle the image files

`tf.data.Dataset.list_files` gives us every image path as a `tf.data.Dataset`, which we can shuffle, split, and transform lazily — this scales to datasets far larger than fits in memory.

We fix a random seed so the shuffling and train/validation split are reproducible across runs.

In [ ]:
import tensorflow as tf
from tensorflow.data import AUTOTUNE

SEED = 42
tf.random.set_seed(SEED)

list_ds = tf.data.Dataset.list_files('temp/*/*.jpg', shuffle=False)
image_count = list_ds.cardinality().numpy()
list_ds = list_ds.shuffle(image_count, seed=SEED, reshuffle_each_iteration=False)
print(f"Total images: {image_count}")

In [ ]:
for f in list_ds.take(5):
  print(f.numpy())

## 3. Split into train and validation sets

We hold out 20% of the images for validation, so we can check the model on data it never trained on.

In [ ]:
VAL_FRACTION = 0.2
val_size = int(image_count * VAL_FRACTION)

train_files = list_ds.skip(val_size)
val_files = list_ds.take(val_size)

print(f"Training images:   {train_files.cardinality().numpy()}")
print(f"Validation images: {val_files.cardinality().numpy()}")

## 4. Read the bounding-box labels

Each `.txt` label file has one line per object: `class x_center y_center width height`. Every image in this dataset has exactly one labeled animal, so we only need the **first line** — and we drop the class id (column 0), keeping only the box (columns 1–4), since this notebook trains a box-only regressor.

In [ ]:
def get_label(file_path):
  txt_file = tf.strings.regex_replace(file_path, r'\.jpg$', '.txt')
  raw = tf.io.read_file(txt_file)
  lines = tf.strings.split(raw, sep="\n")
  values = tf.strings.to_number(tf.strings.split(lines[0], sep=" "))
  box = values[1:]  # drop the class id, keep [x_center, y_center, width, height]
  return box

In [ ]:
sample_box = get_label(tf.convert_to_tensor('temp/buffalo/002.jpg'))
print("Normalized box [x_center, y_center, width, height]:", sample_box.numpy())

## 5. Build the image + label pipeline

`process_path` loads an image, decodes it, resizes it to a fixed size, and pairs it with its box. We map this over both the training and validation file lists.

In [ ]:
IMAGE_WIDTH = 170
IMAGE_HEIGHT = 170

def process_path(file_path):
  label = get_label(file_path)
  img = tf.io.read_file(file_path)
  img = tf.io.decode_jpeg(img, channels=3)
  img = tf.image.resize(img, [IMAGE_HEIGHT, IMAGE_WIDTH])
  return img, label

train_ds = train_files.map(process_path, num_parallel_calls=AUTOTUNE)
val_ds = val_files.map(process_path, num_parallel_calls=AUTOTUNE)

for image, label in train_ds.take(1):
  print("Image shape:", image.numpy().shape)
  print("Label:", label.numpy())

## 5b. Augment the training data

With only ~1,200 training images, the model can easily overfit — memorizing individual photos instead of learning general animal shapes. **Random horizontal flipping** is a cheap way to double the effective variety: we flip the image left-right and mirror the box's `x_center` (`1 - x_center`) to match. We only augment the *training* set — validation images stay unmodified so we measure real performance.

In [ ]:
def random_flip(image, box):
  if tf.random.uniform([]) < 0.5:
    image = tf.image.flip_left_right(image)
    box = tf.stack([1.0 - box[0], box[1], box[2], box[3]])
  return image, box

train_ds = train_ds.map(random_flip, num_parallel_calls=AUTOTUNE)

## 6. Batch, shuffle, and prefetch

`cache`, `shuffle`, `batch`, and `prefetch` keep the GPU fed with data instead of waiting on disk I/O — a standard `tf.data` performance pattern, independent of what model we train.

In [ ]:
BATCH_SIZE = 32

def configure_for_performance(ds):
  ds = ds.cache()
  ds = ds.shuffle(buffer_size=1000, seed=SEED)
  ds = ds.batch(BATCH_SIZE)
  ds = ds.prefetch(buffer_size=AUTOTUNE)
  return ds

train_ds = configure_for_performance(train_ds)
val_ds = configure_for_performance(val_ds)

## 7. Helper: convert boxes to pixel corners and draw them

Our boxes are stored as `(x_center, y_center, width, height)`, normalized to `[0, 1]`. To draw them with OpenCV we need pixel corner coordinates `(x_min, y_min, x_max, y_max)`. We write this conversion once and reuse it for both ground-truth and predicted boxes, later in the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

def to_corners(boxes, image_size):
  """(x_center, y_center, width, height) in [0,1] -> pixel (x_min, y_min, x_max, y_max)."""
  boxes = boxes.copy() * image_size
  x_min = boxes[:, 0] - boxes[:, 2] / 2
  y_min = boxes[:, 1] - boxes[:, 3] / 2
  x_max = boxes[:, 0] + boxes[:, 2] / 2
  y_max = boxes[:, 1] + boxes[:, 3] / 2
  return np.stack([x_min, y_min, x_max, y_max], axis=1).astype(int)

def draw_boxes(images, gt_boxes=None, pred_boxes=None, n=9, cols=3, figsize=(10, 10)):
  """Draw ground-truth boxes in green and predicted boxes in red."""
  gt_corners = to_corners(gt_boxes, IMAGE_WIDTH) if gt_boxes is not None else None
  pred_corners = to_corners(pred_boxes, IMAGE_WIDTH) if pred_boxes is not None else None

  rows = int(np.ceil(n / cols))
  plt.figure(figsize=figsize)
  for i in range(n):
    plt.subplot(rows, cols, i + 1)
    img = images[i].numpy().astype("uint8").copy()
    if gt_corners is not None:
      x0, y0, x1, y1 = gt_corners[i]
      cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
    if pred_corners is not None:
      x0, y0, x1, y1 = pred_corners[i]
      cv2.rectangle(img, (x0, y0), (x1, y1), (255, 0, 0), 2)
    plt.imshow(img)
    plt.axis("off")
  plt.tight_layout()
  plt.show()

## 8. Visualize the ground-truth boxes

In [ ]:
image_batch, label_batch = next(iter(train_ds))
draw_boxes(image_batch, gt_boxes=label_batch.numpy(), n=9)

## 9. Build a transfer-learning model for bounding-box regression

Training a large CNN from scratch on only ~1,200 images is hard — that's a big reason the earlier version of this notebook underperformed. Instead, we reuse **MobileNetV2**, a CNN already trained on 1.4 million ImageNet photos, as a fixed feature extractor: it already knows how to recognize edges, textures, and shapes, so we only need to learn a small regression head on top.

We freeze the pretrained weights (`base_model.trainable = False`) so training only updates our small head — this is fast, needs far less data, and is much less prone to overfitting than training everything from scratch. The output layer uses a `sigmoid` activation, since our box coordinates are always normalized to `[0, 1]`. We also switch from MSE to **Huber loss**, which is less sensitive to the occasional imprecisely-labeled box than plain squared error.

In [ ]:
from tensorflow import keras

base_model = keras.applications.MobileNetV2(
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, 3),
    include_top=False,
    weights='imagenet')
base_model.trainable = False  # keep the pretrained features frozen

# TODO: Design the model that have: 
# Input, 
# Rescaling (MobileNetV2 expects inputs in [-1, 1]), 
# Base Model, 
# GlobalAveragePooling layer, 
# Hidden Layer, 
# Dropout, 
# and Output (box coordinates between [0, 1])
model = ...

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=keras.losses.Huber(),
    metrics=['mse'])

model.summary()

## 10. Train the model

`EarlyStopping` watches the validation loss and stops training once it stops improving, restoring the best-performing weights. This is a simple way to avoid overfitting without having to guess the "right" number of epochs.

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

EPOCHS = 30
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    validation_data=val_ds,
    callbacks=callbacks
)

## 11. Plot the training curves

Comparing training loss to validation loss shows whether the model is still improving, overfitting (training loss keeps dropping while validation loss rises), or underfitting (both losses stay high).

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='validation loss')
plt.xlabel('Epoch')
plt.ylabel('Huber loss')
plt.legend()
plt.title('Training vs. validation loss')
plt.show()

## 12. Evaluate with Intersection over Union (IoU)

MSE tells us how far off the raw numbers are, but it doesn't directly say how well two boxes overlap. **IoU** (Intersection over Union) is the metric object detection actually cares about: the overlapping area of the predicted and true boxes, divided by the area they cover together. IoU = 1 means a perfect match; IoU = 0 means the boxes don't touch at all. Real detectors are graded using IoU thresholds (e.g. "counts as correct if IoU > 0.5") and the mAP (mean average precision) metric built on top of it.

In [ ]:
def iou(boxes_a, boxes_b):
  """IoU between two batches of (x_min, y_min, x_max, y_max) boxes."""
  x0 = np.maximum(boxes_a[:, 0], boxes_b[:, 0])
  y0 = np.maximum(boxes_a[:, 1], boxes_b[:, 1])
  x1 = np.minimum(boxes_a[:, 2], boxes_b[:, 2])
  y1 = np.minimum(boxes_a[:, 3], boxes_b[:, 3])

  intersection = np.maximum(0, x1 - x0) * np.maximum(0, y1 - y0)
  area_a = (boxes_a[:, 2] - boxes_a[:, 0]) * (boxes_a[:, 3] - boxes_a[:, 1])
  area_b = (boxes_b[:, 2] - boxes_b[:, 0]) * (boxes_b[:, 3] - boxes_b[:, 1])
  union = area_a + area_b - intersection

  return intersection / np.maximum(union, 1e-6)

all_ious = []
for image_batch, label_batch in val_ds:
  pred_batch = model.predict(image_batch, verbose=0)
  gt_corners = to_corners(label_batch.numpy(), IMAGE_WIDTH)
  pred_corners = to_corners(pred_batch, IMAGE_WIDTH)
  all_ious.extend(iou(gt_corners, pred_corners))

print(f"Mean IoU on validation set: {np.mean(all_ious):.3f}")

## 13. Visualize predictions vs. ground truth

Green boxes are the ground truth; red boxes are the model's predictions. A tight overlap between the two means a high IoU.

In [ ]:
image_batch, label_batch = next(iter(val_ds))
pred_batch = model.predict(image_batch, verbose=0)

draw_boxes(image_batch, gt_boxes=label_batch.numpy(), pred_boxes=pred_batch, n=9, figsize=(10, 10))

## Part 1 recap: what a single-box regressor can't do yet

This notebook trained a CNN to do **single-object bounding-box regression**: one image in, one box out, using a regression loss and IoU to evaluate the result. That's a useful building block, but production object detectors like **YOLO**, **SSD**, and **Faster R-CNN** go further:

- **Multiple objects per image** — they predict a variable number of boxes, not exactly one.
- **Classification + localization** — each predicted box also gets a class label (and a confidence score), not just coordinates. We threw away the class id in step 4; a real detector would predict it too.
- **Anchor boxes / grid cells** — instead of one `Dense(4)` output, the network predicts boxes relative to a grid of candidate locations and shapes, so it can propose many boxes at once.
- **Non-max suppression (NMS)** — when several candidate boxes overlap the same object, NMS keeps the best one and discards the rest.
- **mAP instead of a single IoU** — performance is summarized across all classes and multiple IoU thresholds.

Let's build the **grid cell** idea ourselves — it's the piece that lets a detector propose more than one box per image.

## Part 2: A Grid-Based Detector (How YOLO's Core Idea Works)

Instead of one `Dense(4)` head that regresses a single box for the whole image, we now divide the image into an `S × S` grid. Each grid cell gets its own tiny prediction: *"is the center of an object here?"* (**objectness**) and, if so, *"what box would you draw?"* This is the mechanism real detectors use to propose many boxes at once — we're building a minimal, single-object version of it on the same dataset, so you can see exactly how it works before reaching for a full YOLO implementation.

We pick `S = 5`, giving a `5 × 5 = 25`-cell grid over each 170×170 image (each cell covers roughly 34×34 pixels).

In [ ]:
GRID_SIZE = 5

def encode_grid_target(box):
  """Turn one normalized (x_center, y_center, w, h) box into an (S, S, 5) grid target.

  Channel 0 is objectness: 1 in the single cell the box's center falls in, 0 elsewhere.
  Channels 1-4 repeat the box, but only the objectness=1 cell's copy is ever used.
  """
  cx, cy = box[0], box[1]
  col = tf.clip_by_value(tf.cast(cx * GRID_SIZE, tf.int32), 0, GRID_SIZE - 1)
  row = tf.clip_by_value(tf.cast(cy * GRID_SIZE, tf.int32), 0, GRID_SIZE - 1)
  cell_index = row * GRID_SIZE + col

  objectness = tf.one_hot(cell_index, GRID_SIZE * GRID_SIZE)
  objectness = tf.reshape(objectness, [GRID_SIZE, GRID_SIZE, 1])

  box_grid = tf.reshape(tf.tile(box, [GRID_SIZE * GRID_SIZE]), [GRID_SIZE, GRID_SIZE, 4])

  return tf.concat([objectness, box_grid], axis=-1)

# Same random_flip augmentation as Part 1, applied to the training split only.
train_ds_grid = train_files.map(process_path, num_parallel_calls=AUTOTUNE)
train_ds_grid = train_ds_grid.map(random_flip, num_parallel_calls=AUTOTUNE)
train_ds_grid = train_ds_grid.map(lambda img, box: (img, encode_grid_target(box)), num_parallel_calls=AUTOTUNE)
train_ds_grid = configure_for_performance(train_ds_grid)

val_ds_grid = val_files.map(process_path, num_parallel_calls=AUTOTUNE)
val_ds_grid = val_ds_grid.map(lambda img, box: (img, encode_grid_target(box)), num_parallel_calls=AUTOTUNE)
val_ds_grid = configure_for_performance(val_ds_grid)

## Visualize a grid target

Before training, let's confirm the encoding makes sense: the **objectness heatmap** should light up exactly one cell — the one containing the animal's center — and that should match where the animal actually is in the image.

In [ ]:
sample_ds = train_files.map(process_path).map(lambda img, box: (img, encode_grid_target(box)))
sample_image, sample_target = next(iter(sample_ds.take(1)))
objectness_map = sample_target.numpy()[..., 0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(sample_image.numpy().astype("uint8"))
axes[0].set_title("Image")
axes[0].axis("off")

axes[1].imshow(objectness_map, cmap="Greens", vmin=0, vmax=1)
axes[1].set_title(f"Objectness grid ({GRID_SIZE}x{GRID_SIZE})")
for i in range(GRID_SIZE):
  for j in range(GRID_SIZE):
    axes[1].text(j, i, f"{objectness_map[i, j]:.0f}", ha="center", va="center")
plt.show()

## Build the grid model

We reuse the same frozen **MobileNetV2** backbone from Part 1 (same reasoning: ~1,200 images isn't enough to train a large CNN from scratch), then swap the head: instead of `Dense(4)`, we output `Dense(S*S*5)` — 5 numbers (1 objectness + 4 box coordinates) for every one of the `S×S` grid cells — and reshape it to `(S, S, 5)`.

Since objectness is a 0/1 value and our box coordinates are already normalized to `[0, 1]`, a single `sigmoid` activation on the whole output works for both at once, keeping the architecture simple.

In [ ]:
# TODO: Use the same frozen MobileNetV2 
base_model_grid = ...

# TODO: Design the model with:
# Input, 
# Rescaling (MobileNetV2 expects inputs in [-1, 1]), 
# Base Model, 
# GlobalAveragePooling layer, 
# Hidden Layer, 
# Dropout, 
# Output (as described above)
# Reshape layer to convert Dense(S*S*5) to (S, S, 5)

grid_model = ...

grid_model.summary()

## Understanding the `(S, S, 5)` output shape

It's easy to misread `(5, 5, 5)` as some kind of 3D box grid — it isn't. The first two `5`s are the spatial grid (5 rows × 5 columns = 25 cells). The **last** `5` is a per-cell vector, not a grid dimension at all:

```
channel 0: objectness   → is the object's center in this cell? (0-1)
channel 1: x_center     → box, only meaningful if objectness ≈ 1
channel 2: y_center
channel 3: width
channel 4: height
```

So each cell carries exactly **one** objectness score plus **one** candidate box — 25 cells × 5 numbers = 125 values total, reshaped to `(5, 5, 5)`. `grid_model_output[2, 3, 0]` asks "how confident is the model that the object's center is in row 2, column 3?", and `grid_model_output[2, 3, 1:5]` is the box it would draw if so.

**One box per cell is a real limitation.** Because each cell only has one 5-value slot, it can only ever describe one object. Two objects whose centers land in *different* cells are fine — each cell proposes its own box independently. But two objects whose centers land in the *same* cell can't both be represented; one gets silently dropped. Real YOLO solves this with **anchor boxes**: instead of `Dense(S*S*5)`, it uses `Dense(S*S*B*5)`, giving each cell `B` separate box "slots" (each tuned to a different box shape), so a cell can propose several boxes instead of just one.

## A loss function for grid + objectness

The loss has two parts:

- **Objectness loss** (binary cross-entropy) — applied to *every* cell, since every cell should learn whether the object's center is there or not.
- **Box loss** (mean squared error) — only applied to the *one responsible cell* per image (masked by the true objectness).

Two details matter for this to actually train well: we **normalize the box loss by the number of responsible cells** (exactly one per image) instead of averaging over all 25 cells — otherwise the box-error gradient gets diluted 25x and the model barely learns to place the box accurately. We also multiply the box loss by `lambda_coord = 5`, a standard YOLO trick that says "getting the box right matters more than getting the objectness score right."

In [ ]:
def grid_loss(y_true, y_pred, lambda_coord=5.0):
  true_obj = y_true[..., 0]
  true_box = y_true[..., 1:]
  pred_obj = y_pred[..., 0]
  pred_box = y_pred[..., 1:]

  objectness_loss = tf.reduce_mean(
      keras.losses.binary_crossentropy(true_obj[..., tf.newaxis], pred_obj[..., tf.newaxis]))

  box_error = tf.reduce_sum(tf.square(true_box - pred_box), axis=-1) * true_obj
  num_objects = tf.reduce_sum(true_obj, axis=[1, 2]) + 1e-6  # exactly one responsible cell per image
  box_loss = tf.reduce_mean(tf.reduce_sum(box_error, axis=[1, 2]) / num_objects)

  return objectness_loss + lambda_coord * box_loss

grid_model.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0005), loss=grid_loss)

## Train the grid model

Same recipe as Part 1: fit on the training grid targets, validate on held-out images, and stop early once validation loss stops improving.

In [ ]:
grid_history = grid_model.fit(
    train_ds_grid,
    epochs=EPOCHS,
    validation_data=val_ds_grid,
    callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
)

## Decode predictions: pick the most confident cell

At inference time, each of the 25 cells produces an objectness score and a candidate box. Since we know there's only one animal per image, we simply keep the cell with the **highest objectness score** and use its box — a simplified stand-in for the non-max suppression a multi-object detector would need.

In [ ]:
def decode_grid_predictions(pred_grid):
  """(batch, S, S, 5) -> boxes (batch, 4), confidence (batch,), using the highest-objectness cell."""
  batch_size = pred_grid.shape[0]
  flat = pred_grid.reshape(batch_size, -1, 5)
  best_cell = np.argmax(flat[..., 0], axis=1)
  best = flat[np.arange(batch_size), best_cell]
  return best[:, 1:], best[:, 0]

grid_ious = []
for image_batch, target_batch in val_ds_grid:
  pred_grid = grid_model.predict(image_batch, verbose=0)
  pred_boxes, _ = decode_grid_predictions(pred_grid)

  true_flat = target_batch.numpy().reshape(target_batch.shape[0], -1, 5)
  true_cell = np.argmax(true_flat[..., 0], axis=1)
  true_boxes = true_flat[np.arange(target_batch.shape[0]), true_cell][:, 1:]

  gt_corners = to_corners(true_boxes, IMAGE_WIDTH)
  pred_corners = to_corners(pred_boxes, IMAGE_WIDTH)
  grid_ious.extend(iou(gt_corners, pred_corners))

print(f"Mean IoU on validation set (grid model):        {np.mean(grid_ious):.3f}")
print(f"Mean IoU on validation set (Part 1 regressor):   {np.mean(all_ious):.3f}")

## Visualize the grid, ground truth, and predictions

We draw the `5×5` grid lines and show the ground-truth box in green and the predicted box in red, along with the model's confidence for the cell it picked.

In [ ]:
def draw_grid_boxes(images, gt_boxes, pred_grids, grid_size=GRID_SIZE, n=9, cols=3, figsize=(10, 10)):
  pred_boxes, confidences = decode_grid_predictions(pred_grids)
  gt_corners = to_corners(gt_boxes, IMAGE_WIDTH)
  pred_corners = to_corners(pred_boxes, IMAGE_WIDTH)

  rows = int(np.ceil(n / cols))
  plt.figure(figsize=figsize)
  step = IMAGE_WIDTH // grid_size
  for i in range(n):
    plt.subplot(rows, cols, i + 1)
    img = images[i].numpy().astype("uint8").copy()

    for k in range(1, grid_size):
      cv2.line(img, (k * step, 0), (k * step, IMAGE_WIDTH), (200, 200, 200), 1)
      cv2.line(img, (0, k * step), (IMAGE_WIDTH, k * step), (200, 200, 200), 1)

    x0, y0, x1, y1 = gt_corners[i]
    cv2.rectangle(img, (x0, y0), (x1, y1), (0, 255, 0), 2)
    x0, y0, x1, y1 = pred_corners[i]
    cv2.rectangle(img, (x0, y0), (x1, y1), (255, 0, 0), 2)

    plt.imshow(img)
    plt.title(f"confidence: {confidences[i]:.2f}")
    plt.axis("off")
  plt.tight_layout()
  plt.show()

image_batch, target_batch = next(iter(val_ds_grid))
pred_grid = grid_model.predict(image_batch, verbose=0)

true_flat = target_batch.numpy().reshape(target_batch.shape[0], -1, 5)
true_cell = np.argmax(true_flat[..., 0], axis=1)
true_boxes = true_flat[np.arange(target_batch.shape[0]), true_cell][:, 1:]

draw_grid_boxes(image_batch, true_boxes, pred_grid, n=9)

## Final wrap-up

The grid model in Part 2 predicts a box *per cell* instead of one box for the whole image — the same core mechanism YOLO uses to propose many boxes at once. But two things still separate it from a real detector:

- **Supervision**: our labels only ever mark *one* responsible cell per image, so the model is never taught what to do when two objects fall in different cells at once. A real detector needs training data with multiple labeled objects per image to learn that.
- **No NMS needed (yet)**: because we only ever pick the single highest-confidence cell, we sidestepped non-max suppression entirely. With multiple true objects, several cells could each confidently propose overlapping boxes, and NMS is what prunes those duplicates down to one box per object.

From here, the **TensorFlow Object Detection API** or **Ultralytics YOLOv8/YOLO11** implement the full version of everything in this notebook — multi-object grids, per-class predictions, anchor boxes, and NMS — and can be fine-tuned on this same dataset using the class ids we discarded in step 4.